# Conditional Granger Causality

This notebook demonstrates **conditional Granger Causality (cGC)** for multivariate systems using `pygc`.

We use the **5-variable AR model from Baccalá & Sameshima (2001)** as a benchmark.
Its ground-truth connectivity is:

$$X_1 \to X_2, \quad X_1 \to X_3, \quad X_1 \to X_4, \quad X_4 \leftrightarrow X_5$$

Conditioning on all remaining channels removes spurious indirect influences that would appear in pairwise (bivariate) GC.

We estimate:
1. **Time-domain conditional GC** — a scalar $F_{j \to i}$ per pair (via `conditional_granger_causality`).
2. **Spectral conditional GC** — a frequency-resolved $\text{cGC}_{j \to i}(f)$ per pair (via `conditional_spec_granger_causality`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from pygc.ar_model import ar_model_baccala
from pygc.granger import conditional_granger_causality, conditional_spec_granger_causality
from pygc.spectral_analysis.fourier import compute_freq, csd_fourier

%matplotlib inline
plt.rcParams.update({'figure.dpi': 100, 'font.size': 12})

## 1. Generate Synthetic Data

The Baccalá 5-variable model has the following equations:

$$
X_1(t) = 0.95\sqrt{2}\,X_1(t{-}1) - 0.9025\,X_1(t{-}2) + \varepsilon_1
$$
$$
X_2(t) = 0.5\,X_1(t{-}2) + \varepsilon_2
$$
$$
X_3(t) = -0.4\,X_1(t{-}3) + \varepsilon_3
$$
$$
X_4(t) = -0.5\,X_1(t{-}2) + 0.25\sqrt{2}\,X_4(t{-}1) + 0.25\sqrt{2}\,X_5(t{-}1) + \varepsilon_4
$$
$$
X_5(t) = -0.25\sqrt{2}\,X_4(t{-}1) + 0.25\sqrt{2}\,X_5(t{-}1) + \varepsilon_5
$$

Ground-truth edges: $X_1 \to X_2$, $X_1 \to X_3$, $X_1 \to X_4$, $X_4 \leftrightarrow X_5$.

In [ ]:
np.random.seed(0)

nvars   = 5
N       = 2048   # samples per trial
ntrials = 50     # trials to average the cross-spectrum
Fs      = 200.0  # sampling rate (Hz)

Y = ar_model_baccala(nvars=nvars, N=N, ntrials=ntrials)
# Y shape: (nvars, N, ntrials)
print('Data shape (channels, samples, trials):', Y.shape)

labels = [f'$X_{i+1}$' for i in range(nvars)]

fig, axes = plt.subplots(nvars, 1, figsize=(12, 8), sharex=True)
t = np.arange(N) / Fs
for i, ax in enumerate(axes):
    ax.plot(t[:300], Y[i, :300, 0], lw=0.8)
    ax.set_ylabel(labels[i])
axes[-1].set_xlabel('Time (s)')
fig.suptitle('Baccalá 5-variable AR model (first 1.5 s, trial 0)')
plt.tight_layout()

## 2. Compute the Cross-Spectral Matrix

We average the Fourier cross-spectrum over trials to obtain $\mathbf{S}(f)$ of shape $(p, p, N/2+1)$.

In [ ]:
f = compute_freq(N, Fs)
nf = len(f)

S = np.zeros((nvars, nvars, nf), dtype=complex)
for trial in range(ntrials):
    for i in range(nvars):
        for j in range(nvars):
            S[i, j] += csd_fourier(Y[i, :, trial], Y[j, :, trial], f=f, Fs=Fs)
S /= ntrials

print('Cross-spectral matrix shape:', S.shape)
print(f'Frequency resolution: {f[1]-f[0]:.3f} Hz  |  max frequency: {f[-1]:.1f} Hz')

## 3. Time-Domain Conditional GC

`conditional_granger_causality` returns a $(p \times p)$ matrix $F$ where $F_{ij} > 0$ indicates
that channel $j$ Granger-causes channel $i$ after conditioning on all other channels.

It runs one Wilson factorization on the full model and one on each $(p{-}1)$-dimensional
reduced model. Pass `n_jobs=-1` to parallelise the reduced models.

In [ ]:
F = conditional_granger_causality(S, f, Fs, Niterations=200, tol=1e-12,
                                  verbose=False, n_jobs=-1)

print('Conditional GC matrix F[i,j] = GC(j -> i):')
print(np.round(F, 4))

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(F, cmap='hot_r', vmin=0)
plt.colorbar(im, ax=ax, label='GC (nats)')
ax.set_xticks(range(nvars))
ax.set_yticks(range(nvars))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_xlabel('Source (j)')
ax.set_ylabel('Target (i)')
ax.set_title('Conditional GC  —  $F_{ij}$ = GC($j \\to i$)')

# Annotate cells
for i in range(nvars):
    for j in range(nvars):
        ax.text(j, i, f'{F[i,j]:.3f}', ha='center', va='center',
                fontsize=9, color='white' if F[i,j] > F.max()*0.5 else 'black')

plt.tight_layout()

The matrix should show strong values at:
- $F_{21}$: $X_1 \to X_2$
- $F_{31}$: $X_1 \to X_3$
- $F_{41}$: $X_1 \to X_4$
- $F_{45}$ and $F_{54}$: bidirectional $X_4 \leftrightarrow X_5$

All other entries should be near zero.

## 4. Spectral Conditional GC

`conditional_spec_granger_causality` returns a $(p \times p \times N_f)$ array so you can see
**which frequency bands** carry each directed influence.

In [ ]:
cGC = conditional_spec_granger_causality(S, f, Fs, Niterations=200, tol=1e-12,
                                         verbose=False, n_jobs=-1)

print('Spectral conditional GC shape (source, target, freq):', cGC.shape)

### 4a. Plot all non-zero directed pairs

In [ ]:
# Pairs with meaningful time-domain GC (threshold at 10% of max)
threshold = 0.1 * F.max()
pairs = [(j, i) for i in range(nvars) for j in range(nvars)
         if i != j and F[i, j] > threshold]

n_pairs = len(pairs)
fig, axes = plt.subplots(1, n_pairs, figsize=(3.5 * n_pairs, 3.5), sharey=False)
if n_pairs == 1:
    axes = [axes]

fmax = 80  # Hz — plot up to 80 Hz
fmask = f <= fmax

for ax, (j, i) in zip(axes, pairs):
    ax.plot(f[fmask], cGC[j, i, fmask], lw=2)
    ax.fill_between(f[fmask], cGC[j, i, fmask], alpha=0.25)
    ax.set_title(f'$X_{j+1} \\to X_{i+1}$')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_xlim([0, fmax])
    ax.set_ylim(bottom=0)

axes[0].set_ylabel('Conditional GC (nats)')
fig.suptitle('Spectral Conditional Granger Causality — Baccalá Model', y=1.02)
plt.tight_layout()

### 4b. Full spectral GC matrix grid

In [ ]:
fig = plt.figure(figsize=(14, 12))
gs  = gridspec.GridSpec(nvars, nvars, hspace=0.4, wspace=0.4)

fmask = f <= fmax
vmax  = cGC[:, :, fmask].max()

for i in range(nvars):        # target
    for j in range(nvars):    # source
        ax = fig.add_subplot(gs[i, j])
        if i == j:
            ax.set_facecolor('#f0f0f0')
            ax.text(0.5, 0.5, labels[i], ha='center', va='center',
                    transform=ax.transAxes, fontsize=13)
            ax.set_xticks([])
            ax.set_yticks([])
        else:
            color = 'C0' if F[i, j] > threshold else 'lightgrey'
            ax.plot(f[fmask], cGC[j, i, fmask], lw=1.5, color=color)
            ax.set_xlim([0, fmax])
            ax.set_ylim(bottom=0, top=vmax * 1.1)
            ax.set_xticks([0, 40, 80])
            ax.tick_params(labelsize=7)
            if j == 0:
                ax.set_ylabel(labels[i], fontsize=10)
            if i == nvars - 1:
                ax.set_xlabel(labels[j], fontsize=10)

fig.suptitle('Spectral cGC matrix  —  row = target, column = source\n'
             'Blue = significant, grey = near zero',
             fontsize=13, y=1.01)
plt.tight_layout()

## 5. Summary

| Edge | Expected | Detected ($F_{ij}$) |
|------|----------|---------------------|
| $X_1 \to X_2$ | ✓ | $F_{21} > 0$ |
| $X_1 \to X_3$ | ✓ | $F_{31} > 0$ |
| $X_1 \to X_4$ | ✓ | $F_{41} > 0$ |
| $X_4 \to X_5$ | ✓ | $F_{54} > 0$ |
| $X_5 \to X_4$ | ✓ | $F_{45} > 0$ |
| All others | ✗ | $F_{ij} \approx 0$ |

Conditioning on all remaining channels correctly eliminates indirect influences
(e.g.  $X_1 \to X_5$ via $X_4$) that would appear spuriously in pairwise GC.

The spectral profiles reveal the characteristic frequency of each influence:
- $X_1$ drives the others near the resonance of its own AR process (~40 Hz).
- The $X_4 \leftrightarrow X_5$ loop has a different spectral signature.